## Code Generator 

#### Goal - To generate High Performance C++ from python code

In [1]:

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import display, Markdown

/Users/kavyahj/AI-Projects/AI-Code-Generator/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
grok_api_key = os.getenv("GROK_API_KEY")
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Grok API Key exists and begins xai-
OpenRouter API Key exists and begins sk-or-


In [4]:
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
grok_url = "https://api.x.ai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)

grok = OpenAI(base_url=grok_url, api_key=grok_api_key)

ollama = OpenAI(api_key="ollama", base_url=ollama_url)

openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)



In [5]:


models = ["gpt-5", "claude-sonnet-4-5-20250929", "grok-4", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b",]

clients = {"gpt-5": openai, "claude-sonnet-4-5-20250929": anthropic, "grok-4": grok, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [6]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '25.5.0',
  'version': 'Darwin Kernel Version 25.5.0: Mon Apr 27 20:41:19 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8122',
  'kernel': '25.5.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin25.5.0'},
 'package_managers': ['xcode-select (CLT)', 'brew'],
 'cpu': {'brand': 'Apple M3',
  'cores_logical': 8,
  'cores_physical': 8,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'g++': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'clang': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

In [34]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model="gpt-5", messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

You do not need to install anything. Your Mac already has a C++ compiler: Apple Clang 21 (via the Xcode Command Line Tools). You can compile and run main.cpp directly.

For best general runtime performance without breaking standards, use high optimization, ThinLTO, and native CPU tuning. In your Python snippet, set:

compile_command = ["clang++", "-std=c++20", "-O3", "-flto=thin", "-mcpu=native", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

Notes:
- If you ever see an error about -mcpu=native on your setup (unlikely), just remove that flag.
- If you can tolerate less strict floating-point semantics for potentially more speed, you can replace -O3 with -Ofast.

In [7]:
compile_command = ["clang++", "-std=c++20", "-O3", "-flto=thin", "-mcpu=native", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [8]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
Do NOT use #include <bits/stdc++.h> — use only standard headers compatible with Apple Clang.
"""

def user_prompt(python_code):
    return f"""
    Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
    The system information is:
    {system_info}
    Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
    {compile_command}
    Respond only with C++ code.
    Python code to port:

    ```python
    {python_code}
    ```
    """

In [9]:
def message_for(python_code):
    return [
        {"role":"system", "content":system_prompt},
        {"role" : "user", "content":user_prompt(python_code)}
    ]

In [10]:
def write_output(cpp_code):
    with open("main.cpp" , "w" , encoding = "utf-8") as f:
        f.write(cpp_code)

In [11]:
def port(model, python_code):
    client = clients[model]
    reasoning_effort = "high" if "gpt" in model else None
    response = client.chat.completions.create(model=model, messages=message_for(python_code), reasoning_effort=reasoning_effort)
    res = response.choices[0].message.content
    res = res.replace('```cpp', '').replace('```', '')
    write_output(res)
    return res


In [12]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [13]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [16]:
run_python(pi)

'Result: 3.141592656089\nExecution Time: 13.639810 seconds\n'

In [42]:
import subprocess
def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [44]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Python(42787) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
